# 07g — Train & Evaluate (Capacity Revision, Pooled Multi-City)

Sixth parallel branch alongside `07` (spatial k-fold), `07b` (bootstrap),
`07c` (60/20/20 random repeats), `07d` (70:30, no val), and `07e`
(fixed-composition CV, joint label x city stratification). Pooled
multi-city data (all cities in `configs/paths.yaml`, from `01`-`05`'s
combined pipeline), PLAIN random 70/15/15 split (`train._stratified_split`,
stratified by label only, NOT by city) via `train.run_scenario_random_repeats`
-- not `07e`'s city-aware `run_scenario_cv_repeats`.

**Updated to the current pipeline schema.** This branch originally
predated the 4-city `paths.yaml` migration and the retirement of
`05b_dataset_assembly_pooled.ipynb` -- it used to read
`paths_cfg["per_city"]["bogor"]["base_dir"]` / a `"combined"` sub-section
(neither exists anymore) and `PooledDualGraphDataset`/`build_feature_table_pooled`
(both keyed by a `uid` column and per-row `svg_dir`/`tvg_dir` columns that
`05b` used to add -- `05` now absorbs that job and writes ONE combined
index/one `SVG_DIR`/`TVG_DIR`, with a globally city-prefixed `point_id`
instead). Every cell below now reads `paths_cfg["interim_dir"]` /
`["processed_dir"]` / `["outputs_dir"]` directly (same top-level keys
`05`/`06` already use) and uses the plain, single-directory
`DualGraphDataset`/`build_feature_table` -- point_id alone is globally
unique now, no `uid` needed.

**`primary_metric` defaults to accuracy now.** `src/train.py`'s
`train_one_fold` used to default to PR-AUC-driven checkpoint selection /
early-stopping / LR-scheduling; that default is now `"accuracy"` for
every branch (this notebook's `eval_capacity_revision.yaml` doesn't set
`primary_metric` explicitly, so it inherits this new default rather than
opting in via a key). The per-epoch verbose print and the persisted
history JSON both surface `val_accuracy` accordingly.

**Per-point raw test predictions are recorded automatically.** Every
`train_one_fold` call with a `history_path` (every scenario/repeat below
has one) now also writes a sibling `..._test_predictions.json`: one
record per test point (`point_id`, `label`, `prob`, `pred`,
`category` = TP/TN/FP/FN at the chosen threshold). A new cell near the
end of this notebook aggregates all of them into one CSV across every
scenario/repeat, for direct point-by-point comparison.

**Why this branch exists (capacity, unchanged from the original design):**
several earlier runs (07b/07c/07e-style, `hidden_dim=128, dropout=0.45`,
`fusion_dim` equal to `hidden_dim`, head hidden width hardcoded at 32)
did not look good. This branch reconsiders every dimension that could be
quietly bottlenecking the graph representations on the way to the final
logit, rather than assuming the architecture itself was fine and only
the split/eval scheme needed changing:

- **`fusion_dim = 256 = hidden_dim*2`** (was `128 == hidden_dim`) -- the
  SVG/TVG encoder's own readout width is `hidden_dim*2` (pool + anchor
  concat, see `models.py`'s `_pool_and_anchor`). A `fusion_dim` equal to
  `hidden_dim` forces a 2x compression right at the encoder-to-fusion
  projection, before the classifier head even sees the representation.
- **`head_hidden = 256`** (was hardcoded `32` in `models.ClassifierHead`,
  never overridden by any prior notebook). 32 was a far sharper
  bottleneck than `fusion_dim` itself -- scenario C/E's concatenated head
  input is `fusion_dim*2 = 512`, so `512 -> 32 -> 1` was throwing away
  far more than the fusion projection ever did.
- **`dropout = 0.3`**, for BOTH the encoder (SVG/TVG GATv2Conv stacks)
  and the classifier head. 0.45 (07b/07c/07e) looked worse, not better;
  0.3-0.35 is the range being retried here, at the low end.
- **Categorical embedding dims doubled**: `cat_embed_dim` 2->4,
  `building_type_embed_dim` 8->16, `highway_embed_dim` 4->8.
- **Encoder depth left at `svg_layers=tvg_layers=2`** -- small graphs,
  more message-passing layers risks over-smoothing rather than fixing a
  real bottleneck.
- **`head_depth` fixed at `"mlp2"` only** (one hidden layer + linear
  output). Not swept against `"linear"` this run.
- **`epoch_cap = 100`**, `warmup_epochs=0`, `patience=100` (structurally disables early stopping -- see the config-setup cell), `lr_patience=3`, `lr=1e-3`, `weight_decay=1e-3`, `head_dropout=0.5`, `test_frac=0.10`.
- **Fixed 0.5 decision threshold** -- this run isolates the
  architecture/capacity variables above, not threshold policy.

**5 repeats** (`n_repeats: 5`).

**No formal significance testing here** -- same caveat as
`07b`/`07c`/`07d`/`07e`: Wilcoxon/Nadeau-Bengio are built for paired,
correlated k-fold scores; random re-splits aren't that. Descriptive
aggregates (mean +/- std across the 5 repeats) only.

GPU recommended.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched src/ files until pushed to GitHub.
# Skip this cell once the repo itself is updated -- needs the revised
# train.py (accuracy-default primary_metric + per-point raw test
# predictions) and models.py, plus graph_datasets.py, unified_graph.py,
# baseline_features.py, evaluate.py, plot_history.py.
from google.colab import files
import shutil

print("Upload train.py, models.py, graph_datasets.py, unified_graph.py, "
      "baseline_features.py, evaluate.py, plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval_capacity_revision.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_capacity_revision.yaml") as f:
    model_cfg = yaml.safe_load(f)

# paths.yaml (current 4-city schema): interim_dir/processed_dir/outputs_dir
# are COMBINED (not per-city) roots at the top level -- every city's 01-04
# output already lands in ONE shared tree, disambiguated by the
# city-prefixed point_id/filename, not by directory. Same top-level keys
# 05/06 already read; no per-city base_dir, no "combined" sub-section.
CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
COMBINED_PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# separate checkpoint/metrics dirs from every other 07 branch
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_capacity_revision"
METRICS_DIR = OUTPUTS_DIR / "metrics_capacity_revision"
for d in [CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_DEPTH = model_cfg.get("head_depth", "mlp2")  # fixed, NOT swept -- see notebook intro
config = {"batch_size": eval_cfg.get("batch_size") or 128,
          "epoch_cap": 150,
          "warmup_epochs": eval_cfg.get("warmup_epochs", 30),
          "patience": 150,
          # lr_patience/lr/weight_decay/head_dropout/test_frac/num_workers
          # are HARDCODED below (not read from eval_cfg/model_cfg) --
          # deliberate, explicit hyperparameter overrides for this run.
          "lr_patience": 10,
          "lr": 1e-3,
          "weight_decay": 1e-3,
          "fusion_dim": model_cfg.get("fusion_dim") or 128,
          "head_hidden": model_cfg.get("head_hidden") or 32,
          "head_dropout": 0.5,
          "val_frac": eval_cfg.get("val_frac") or 0.15,
          "test_frac": 0.10,
          "label_col": eval_cfg.get("label_col") or "label",
          "target_pos_frac": eval_cfg.get("target_pos_frac"),
          "threshold_method": eval_cfg.get("threshold_method") or "fixed",
          "threshold_fn_cost": eval_cfg.get("threshold_fn_cost") or 10.0,
          "threshold_fp_cost": eval_cfg.get("threshold_fp_cost") or 1.0,
          "num_workers": 8,
          "use_amp": eval_cfg.get("use_amp", True),
          # NOT set in eval_capacity_revision.yaml -- train.py's
          # train_one_fold now defaults PRIMARY_METRIC to "accuracy" on
          # its own when this key is absent, so this line is here only to
          # make that inherited default visible/printable below, not
          # because it changes any behavior.
          "primary_metric": eval_cfg.get("primary_metric", "accuracy")}
N_REPEATS = 3

train_frac = 1 - config["val_frac"] - config["test_frac"]
print("Device:", device, "| n_repeats:", N_REPEATS, "| head_depth:", HEAD_DEPTH, "(fixed, not swept)")
print("primary_metric:", config["primary_metric"], "(train.py's default -- not set explicitly in this config)")
print("Split:", f"{train_frac:.0%}/{config['val_frac']:.0%}/{config['test_frac']:.0%}",
      "(train/val/test), stratified by", config["label_col"], "ONLY -- plain random, NOT by city (unlike 07e)")
print("Capacity: hidden_dim=", model_cfg.get("hidden_dim"), " fusion_dim=", config["fusion_dim"],
      " head_hidden=", config["head_hidden"], " dropout=", model_cfg.get("dropout"),
      " head_dropout=", config["head_dropout"], sep="")
print("Epoch budget: warmup=", config["warmup_epochs"], " patience=", config["patience"],
      " epoch_cap=", config["epoch_cap"], sep="")

In [ ]:
import json
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models

SVG_DIR = COMBINED_PROCESSED_DIR / "svg_graphs"
TVG_DIR = COMBINED_PROCESSED_DIR / "tvg_graphs"
INDEX_PATH = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
index_df = pd.read_parquet(INDEX_PATH)
assert "city" in index_df.columns, (
    f"'{INDEX_PATH}' has no 'city' column -- this notebook needs 05's combined, "
    "multi-city dataset_index.parquet (all cities pooled into one index/one "
    "SVG_DIR/TVG_DIR, city-prefixed point_id), not a single-city index.")

# DualGraphDataset, not PooledDualGraphDataset -- the latter is vestigial
# from the old (deleted) 05b_dataset_assembly_pooled.ipynb design (per-row
# svg_dir/tvg_dir columns, a separate 'uid' column). Current 01-05 already
# gives every point_id a global city prefix (bog_/war_/kra_/som_) and
# writes every city's graphs into ONE shared SVG_DIR/TVG_DIR, so the plain
# single-directory dataset class (same one 05/06 already use) is correct
# here too -- point_id itself doubles as the globally-unique id.
dataset = ds.DualGraphDataset(index_df, SVG_DIR, TVG_DIR)
print(f"Dataset: {len(dataset)} points (pooled, capacity-revision branch)")
print(index_df.groupby("city")["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

# Unified vocab sizes, read from the post-04b cache -- same convention as
# 05/06: any city's cache holds the identical unified vocab post-04b, so
# the first city in CITIES is as good as any other to read from. Confirm
# every OTHER city's cache agrees, catching a partially-run 04b before it
# silently misaligns the shared embedding table.
_ref_cache_dir = INTERIM_DIR / "osm_cache" / CITIES[0]
with open(_ref_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(_ref_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB_SIZE = len(json.load(f))
for city in CITIES[1:]:
    cache_dir = INTERIM_DIR / "osm_cache" / city
    with open(cache_dir / "highway_vocab.json") as f:
        hw_n = len(json.load(f))
    with open(cache_dir / "building_type_vocab.json") as f:
        bt_n = len(json.load(f))
    assert hw_n == HIGHWAY_VOCAB_SIZE and bt_n == BUILDING_TYPE_VOCAB_SIZE, (
        f"{city}'s vocab cache ({hw_n} highway / {bt_n} building_type) disagrees with "
        f"{CITIES[0]}'s ({HIGHWAY_VOCAB_SIZE} / {BUILDING_TYPE_VOCAB_SIZE}) -- run "
        f"04b_vocab_unification for every city before trusting this notebook.")
print(f"Unified vocab (post-04b, all {len(CITIES)} cities agree): "
      f"highway={HIGHWAY_VOCAB_SIZE}, building_type={BUILDING_TYPE_VOCAB_SIZE}")

svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2,
                   cat_embed_dim=model_cfg.get("cat_embed_dim", 4))
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   building_type_vocab=BUILDING_TYPE_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=model_cfg.get("building_type_embed_dim", 16),
                   highway_embed_dim=model_cfg.get("highway_embed_dim", 8))
print("svg_kwargs:", svg_kwargs)
print("tvg_kwargs:", tvg_kwargs)

### Scenario A -- SVG only

In [ ]:
key = "A"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("A", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results = {key: results}
print(f"  {len(results)} repeat-runs complete.")

### Scenario B -- TVG only

In [ ]:
key = "B"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("B", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario C -- dual graph (concat)

In [ ]:
key = "C"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("C", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario D -- dual graph (late fusion)

In [ ]:
key = "D"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("D", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario E -- dual graph (cross-attention)

In [ ]:
key = "E"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("E", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario F -- unified merged graph

In [ ]:
key = "F"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("F", HEAD_DEPTH, use_ablation=False, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Ablation B+ through F+

Same rationale as `07`/`07b`/`07c`/`07e`: one cell per scenario, split
out for independent run/monitor/interrupt.

#### Ablation B+

In [ ]:
key = "B_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("B", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation C+

In [ ]:
key = "C_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("C", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation D+

In [ ]:
key = "D_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("D", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation E+

In [ ]:
key = "E_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("E", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

#### Ablation F+

In [ ]:
key = "F_ablation"
print(f"\n=== {key} ({HEAD_DEPTH}) ===")
results = tr.run_scenario_random_repeats("F", HEAD_DEPTH, use_ablation=True, dataset=dataset,
                                          n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                          tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
all_results[key] = results
print(f"  {len(results)} repeat-runs complete.")

### Scenario G -- XGBoost, separate path

Tabular flattened features via `baseline_features.build_feature_table`
(single shared `SVG_DIR`/`TVG_DIR`, keyed by `point_id` -- NOT the old
`build_feature_table_pooled`, which read per-row `svg_dir`/`tvg_dir`
columns and keyed by `uid`, both `05b`-era artifacts that no longer
exist), split via `train._stratified_split` directly -- plain
label-only stratification, matching this branch's "just random" 70/15/15
scheme, NOT `07e`'s `_stratified_split_by_city` (G has no GNN encoder, so
it never goes through `train_one_fold`/`run_scenario_random_repeats` --
same standalone-path convention every other 07 branch uses for G).

In [ ]:
import baseline_features
from xgboost import XGBClassifier

feat_table = baseline_features.build_feature_table(index_df["point_id"].tolist(), SVG_DIR, TVG_DIR, torch)
feat_table = feat_table.merge(index_df[["point_id", "label", "city"]], on="point_id")
feature_cols = [c for c in feat_table.columns if c not in ["point_id", "label", "city"]]


def _g_category(true, pred):
    if true == 1 and pred == 1:
        return "TP"
    if true == 0 and pred == 0:
        return "TN"
    if true == 0 and pred == 1:
        return "FP"
    return "FN"


def _g_raw_predictions(point_ids, trues, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return [
        {"point_id": pid, "label": int(t), "prob": float(p), "pred": int(pred),
         "category": _g_category(int(t), int(pred))}
        for pid, t, p, pred in zip(point_ids, trues, probs, preds)
    ]


g_results = []
for repeat in range(N_REPEATS):
    repeat_seed = 42 + repeat
    # SAME split scheme as A-F at this repeat index -- plain label-stratified
    # random re-split (train._stratified_split), NOT city-stratified.
    train_, val_, test = tr._stratified_split(
        feat_table, config["label_col"], config["val_frac"], config["test_frac"],
        repeat_seed, config.get("target_pos_frac"))

    clf = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="aucpr", random_state=42)
    clf.fit(train_[feature_cols], train_["label"])

    # Always score val (not just for adaptive thresholding) -- needed to
    # report a full val-split metric suite alongside test below, same as
    # train_one_fold now does for every GNN scenario.
    val_prob = clf.predict_proba(val_[feature_cols])[:, 1]
    if config["threshold_method"] == "fixed":
        chosen_threshold = 0.5
        threshold_method_used = "fixed"
    else:
        chosen_threshold, threshold_method_used, _ = ev.find_optimal_threshold(
            val_["label"].values, val_prob, method=config["threshold_method"],
            fn_cost=config["threshold_fn_cost"], fp_cost=config["threshold_fp_cost"])

    val_metrics = ev.compute_metrics(val_["label"].values, val_prob, threshold=chosen_threshold)
    val_metrics["threshold_method"] = threshold_method_used

    prob = clf.predict_proba(test[feature_cols])[:, 1]
    metrics = ev.compute_metrics(test["label"].values, prob, threshold=chosen_threshold)
    metrics["threshold_method"] = threshold_method_used
    g_results.append({"repeat": repeat, "n_train": len(train_), "n_val": len(val_), "n_test": len(test),
                       **metrics})

    # Per-point raw predictions + combined val/test metrics file, same
    # shape/naming convention train_one_fold uses for every GNN scenario.
    g_history_dir = CHECKPOINT_DIR / "G_history"
    g_history_dir.mkdir(parents=True, exist_ok=True)

    test_predictions = _g_raw_predictions(test["point_id"].values, test["label"].values, prob, chosen_threshold)
    with open(g_history_dir / f"repeat{repeat}_test_predictions.json", "w") as f:
        json.dump(test_predictions, f, indent=1)

    val_predictions = _g_raw_predictions(val_["point_id"].values, val_["label"].values, val_prob, chosen_threshold)
    val_test_payload = {
        "val": {"metrics": val_metrics, "predictions": val_predictions},
        "test": {"metrics": metrics, "predictions": test_predictions},
    }
    with open(g_history_dir / f"repeat{repeat}_val_test_metrics.json", "w") as f:
        json.dump(val_test_payload, f, indent=1)

all_results["G"] = g_results
print(f"Scenario G: {len(g_results)} repeat-runs complete.")

## Aggregate + report every scenario

In [ ]:
agg_rows = []
for key, results in all_results.items():
    agg = ev.aggregate_fold_results(results)
    row = {"scenario": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(METRICS_DIR / "all_scenarios_summary_capacity_revision.csv", index=False)
display(agg_df)

## Threshold diagnostics

In [ ]:
threshold_rows = []
for key, results in all_results.items():
    method_counts = ev.summarize_categorical_field(results, "threshold_method")
    thresh_mean, thresh_std = ev.aggregate_fold_results(results).get("threshold_used", (float("nan"), float("nan")))
    threshold_rows.append({"scenario": key, "threshold_mean": thresh_mean, "threshold_std": thresh_std,
                            "methods_used": method_counts})

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(METRICS_DIR / "threshold_diagnostics_capacity_revision.csv", index=False)
display(threshold_df)

## Epoch-level diagnostics

Per-repeat training history saved under
`CHECKPOINT_DIR/{tag}_history/repeat{N}.json`, same JSON shape as every
other branch -- now including `val_accuracy` alongside `val_pr_auc`/
`val_auroc` (added when `primary_metric` became accuracy-driven by
default).

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = CHECKPOINT_DIR / f"A_{HEAD_DEPTH}_history" / "repeat0.json"
history = json.loads(history_path.read_text())

epochs = [h["epoch"] for h in history]
best_epoch = max(range(len(history)), key=lambda i: history[i]["val_pr_auc"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs, [h["train_loss"] for h in history], label="train_loss", color="tab:blue")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train_loss", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(epochs, [h.get("val_accuracy") for h in history], label="val_accuracy (selection metric)", color="tab:red")
ax2.plot(epochs, [h["val_pr_auc"] for h in history], label="val_pr_auc", color="tab:orange")
ax2.plot(epochs, [h["val_auroc"] for h in history], label="val_auroc", color="tab:green")
ax2.axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch ({best_epoch})")
ax2.set_ylabel("val metric")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
plt.title(f"repeat0 history ({history_path.name}, primary_metric={config['primary_metric']})")
plt.tight_layout()
plt.show()

## Per-point raw predictions (TP/TN/FP/FN), aggregated across every scenario/repeat

`train_one_fold` (and scenario G's own loop above) each write a
`..._test_predictions.json` file per (scenario, repeat) -- one record
per test point (`point_id`, `label`, `prob`, `pred`, `category`). This
cell walks every `{tag}_history/repeat{N}_test_predictions.json` under
`CHECKPOINT_DIR`, tags each row with its `scenario`/`repeat`, and
concatenates everything into ONE CSV so specific points can be compared
directly across scenarios (e.g. "which points does scenario A always get
wrong that F gets right") rather than only via aggregate metrics.

In [ ]:
import re

raw_pred_rows = []
for history_dir in CHECKPOINT_DIR.glob("*_history"):
    tag = history_dir.name[: -len("_history")]
    for pred_file in history_dir.glob("repeat*_test_predictions.json"):
        m = re.match(r"repeat(\d+)_test_predictions\.json", pred_file.name)
        repeat_idx = int(m.group(1)) if m else -1
        records = json.loads(pred_file.read_text())
        for r in records:
            raw_pred_rows.append({"scenario": tag, "repeat": repeat_idx, **r})

if raw_pred_rows:
    raw_pred_df = pd.DataFrame(raw_pred_rows)
    raw_pred_df.to_csv(METRICS_DIR / "raw_test_predictions_all_scenarios.csv", index=False)
    print(f"Aggregated {len(raw_pred_df)} per-point predictions across "
          f"{raw_pred_df['scenario'].nunique()} scenario/ablation tags, "
          f"{raw_pred_df['repeat'].nunique()} repeats each.")
    display(raw_pred_df.groupby("scenario")["category"].value_counts().unstack(fill_value=0))
else:
    print("No test_predictions.json files found yet -- run scenario cells above first.")

In [ ]:
print("Capacity-revision branch complete.")
print("Every scenario A-G (including F) trained with fusion_dim=256, head_hidden=256,")
print("dropout=0.3, doubled embedding dims, epoch_cap=100, on a plain random 70/15/10")
print(f"pooled multi-city split. primary_metric={config['primary_metric']} drove checkpoint")
print("selection/early-stopping/LR-scheduling. See METRICS_DIR/raw_test_predictions_all_scenarios.csv")
print("for point-by-point TP/TN/FP/FN comparisons across every scenario/repeat.")